# Day 35: Project 04, Inventory Manager

**Today's question:** how do we go from a written brief to a working, tested, class-based program?

Brief: `projects/04-inventory-manager/README.md`. This notebook only shows the *techniques*; the project itself is
written in `.py` files.

## 1. From brief to design

> *Zara runs a small corner shop near campus. She wants a console program to keep track of her **products**: each
> has a **name**, a **price** and a **quantity** in stock. She **restocks** products, **sells** them, wants to **find**
> a product by name, see which products are **running low**, and know the **total value** of her stock.*

| From the brief | Becomes |
|----------------|---------|
| product, name, price, quantity | class `Product` with three attributes |
| restock, sell, value of one product's stock | `Product` methods |
| the whole stock, find by name, running low, total value | class `Inventory`: it **has** many products |
| menu, asking, printing | `main.py` only: classes never call `input()` |
| saving to a file (Should) | `storage.py` only |

## 2. The starter files

```
starter/
├── inventory.py        Product and Inventory: __init__ and restock are done, the rest are TODOs
├── main.py             the menu loop (runs already); some options are TODOs
├── test_inventory.py   3 tests pass, 7 fail until the TODOs are done; add your own at the bottom
├── storage.py          (Should) save/load as text: TODOs
└── inventory.txt       sample stock, one product per line: name,price,quantity
```

Commands (in the terminal, inside the starter folder): `uv run pytest -q` and `uv run main.py`.

## 3. The test-first loop

Watch the loop on the one method the starter already gives you, `restock`. Here its body is still a TODO:

In [ ]:
class Product:
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    def restock(self, amount):
        pass   # TODO


def test_restock_adds_to_quantity():
    pen = Product("Pen", 0.75, 3)
    pen.restock(10)
    assert pen.quantity == 13

**Step 1–2:** run the test and watch it fail (red). The failure tells us what is missing.

In [ ]:
test_restock_adds_to_quantity()

**Step 3:** write the smallest code that makes it pass: check first, change second.

In [ ]:
class Product:
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    def restock(self, amount):
        if amount <= 0:
            raise ValueError(f"Restock amount must be at least 1, got {amount}.")
        self.quantity += amount

**Step 4:** run again (green), then move to the next TODO.

In [ ]:
test_restock_adds_to_quantity()
print("passed")

In the project, pytest runs *all* tests at once: `uv run pytest -q`. Fix the **first** failure, run again, repeat.

## 4. Looking things up by lowercase name

A user may type `pen`, `Pen` or ` PEN `. If we store every key in lowercase and look up in lowercase, they all match.
`.get()` returns `None` instead of crashing when the key is missing (Day 08).

In [ ]:
prices = {}
prices["Glue stick".lower()] = 1.25
prices["Pen".lower()] = 0.75
print(prices)

typed = "  PEN "
print(prices.get(typed.strip().lower()))
print(prices.get("scissors"))

## 5. Sorting objects by one of their attributes

`sorted()` needs to know *what* to compare. Give it a small function that returns that value for one object (Day 21).

In [ ]:
def price_of(product):
    return product.price


stock = [Product("Notebook", 2.5, 40), Product("Pen", 0.75, 3), Product("Stapler", 6.0, 2)]
for product in sorted(stock, key=price_of):
    print(product.name, product.price)

🤔 **Predict:** how would you get them sorted by quantity instead, and only the ones with 5 or fewer left?

## 6. The menu pattern: helpers ask, classes check, the menu reports

The starter's `ask_int` keeps asking until the answer is a whole number that is big enough:

In [ ]:
def ask_int(prompt, minimum):
    """Ask until the user types a whole number that is at least `minimum`."""
    while True:
        text = input(prompt).strip()
        try:
            number = int(text)
        except ValueError:
            print(f"'{text}' is not a whole number.")
            continue
        if number < minimum:
            print(f"Please enter {minimum} or more.")
        else:
            return number


pen = Product("Pen", 0.75, 3)
amount = ask_int("How many pens to add? ", 1)
pen.restock(amount)
print("Pens in stock:", pen.quantity)

The **classes** raise `ValueError` with a clear message. The **menu** catches it and shows it, so the program never
crashes on a bad request:

In [ ]:
for amount in [4, 0, -2]:
    try:
        pen.restock(amount)
        print(f"Restocked {amount}. Now {pen.quantity}.")
    except ValueError as error:
        print("Sorry:", error)

## 7. Plan for today

1. `Product` TODOs 1–5 → Product tests green.
2. `Inventory` TODOs 6–12 → all starter tests green.
3. Menu TODOs M1–M5 → try every option by hand, including bad input.
4. At least 4 tests of your own (2 with `pytest.raises`).
5. Should: `storage.py`, saving on quit, sales log.

Run the tests after **every** TODO.